In [ ]:
import os

# A mappa neve
folder = "datafiles"

# Fájlok és mappák listázása
files = os.listdir(folder)

print(files)


In [ ]:
import pandas as pd

for file in files:
    name = file.split('.')[0]
    print(f"\n{name}:")

    globals()[f"{name}"] = pd.read_csv(f"datafiles/{file}")
    print(globals()[f"{name}"].sample(5))

In [ ]:
import pandas as pd

# Csak a befejezett versenyek (pozíció nem null)
results_finished = results[results['positionText'] != 'R'].copy()
results_finished['position'] = pd.to_numeric(results_finished['position'], errors='coerce')

# Qualifying és results összekapcsolása
df = qualifying.merge(results_finished, on=['raceId', 'driverId', 'constructorId'], how='inner')

# Verseny infók hozzáadása
df = df.merge(races[['raceId', 'year', 'name', 'circuitId']], on='raceId', how='left')

# Pálya infók hozzáadása
df = df.merge(circuits[['circuitId', 'name', 'country']], on='circuitId', how='left', suffixes=('_race', '_circuit'))

# Versenyző infók
df = df.merge(drivers[['driverId', 'forename', 'surname', 'nationality']], on='driverId', how='left')
df['driver_name'] = df['forename'] + ' ' + df['surname']
df['driver_nationality'] = df['nationality']

# Versenyző tapasztalata (eddigi versenyek száma AZ AKTUÁLIS VERSENY ELŐTT)
# Először hozzáadjuk a dátumot a results táblához
results_with_date = results.merge(races[['raceId', 'date']], on='raceId', how='left')

# Minden versenyző-verseny kombinációhoz számoljuk az előző versenyek számát
driver_exp_list = []
for idx, row in df.iterrows():
    current_driver = row['driverId']
    current_race_id = row['raceId']
    current_date = races[races['raceId'] == current_race_id]['date'].iloc[0]
    
    # Összes korábbi verseny száma
    previous_races = results_with_date[
        (results_with_date['driverId'] == current_driver) & 
        (results_with_date['date'] < current_date)
    ]
    
    driver_exp_list.append({
        'raceId': current_race_id,
        'driverId': current_driver,
        'driver_all_races': len(previous_races)
    })

driver_exp_df = pd.DataFrame(driver_exp_list)
df = df.merge(driver_exp_df, on=['raceId', 'driverId'], how='left')

# Tapasztalat kategória
df['experience_category'] = pd.cut(df['driver_all_races'], 
                                     bins=[-1, 10, 50, 200, 1000], 
                                     labels=['Rookie', 'Junior', 'Experienced', 'Veteran'])

# Konstruktőr infók
df = df.merge(constructors[['constructorId', 'name', 'nationality']], on='constructorId', how='left', suffixes=('', '_constructor'))
df['constructor_name'] = df['name']
df['constructor_nationality'] = df['nationality_constructor']

# Pozíció változás számítás
df['position_change'] = df['position_x'] - df['position_y']  # position_x=quali, position_y=race

# Évtized kategória
df['decade'] = (df['year'] // 10) * 10

# Korszakok (era) manuális kategorizálás
def get_era(year):
    if year <= 1960:
        return 'Early F1 (1950-1960)'
    elif year <= 1983:
        return 'Pre-Turbo (1961-1983)'
    elif year <= 1988:
        return 'Turbo Era (1984-1988)'
    elif year <= 2013:
        return 'V8/V10 Era (1989-2013)'
    elif year <= 2021:
        return 'Hybrid Era (2014-2021)'
    else:
        return 'Ground Effect Era (2022-)'

df['era'] = df['year'].apply(get_era)

# Grid pozíció kategória
df['grid_category'] = pd.cut(df['grid'], bins=[0, 3, 10, 30], labels=['Top3', 'Midfield', 'Back'])

# Kontinens (egyszerűsített)
continent_map = {
    'UK': 'Europe', 'Italy': 'Europe', 'Germany': 'Europe', 'France': 'Europe', 
    'Spain': 'Europe', 'Monaco': 'Europe', 'Belgium': 'Europe', 'Austria': 'Europe',
    'Netherlands': 'Europe', 'Portugal': 'Europe', 'Hungary': 'Europe', 'Russia': 'Europe',
    'USA': 'Americas', 'Brazil': 'Americas', 'Canada': 'Americas', 'Mexico': 'Americas',
    'Argentina': 'Americas',
    'Japan': 'Asia', 'China': 'Asia', 'Malaysia': 'Asia', 'Singapore': 'Asia',
    'Korea': 'Asia', 'Bahrain': 'Asia', 'UAE': 'Asia', 'Saudi Arabia': 'Asia',
    'Australia': 'Oceania',
    'South Africa': 'Africa'
}
df['continent'] = df['country'].map(continent_map).fillna('Other')

# Pályák DNF aránya (historikus, az aktuális verseny ELŐTTI adatok alapján)
results_with_race = results.merge(races[['raceId', 'circuitId', 'date']], on='raceId')
results_with_race['is_dnf'] = (results_with_race['positionText'] == 'R').astype(int)

# Minden versenyhez kiszámoljuk az addig történt DNF arányt azon a pályán
circuit_dnf_history = []
for circuit_id in results_with_race['circuitId'].unique():
    circuit_races = results_with_race[results_with_race['circuitId'] == circuit_id].sort_values('date')
    
    for i, race_id in enumerate(circuit_races['raceId'].unique()):
        # Csak az EZT MEGELŐZŐ versenyek számítanak
        previous_races = circuit_races[circuit_races['raceId'].isin(
            circuit_races['raceId'].unique()[:i]
        )]
        
        if len(previous_races) > 0:
            dnf_rate = previous_races['is_dnf'].mean()
        else:
            dnf_rate = None  # Első verseny a pályán
        
        circuit_dnf_history.append({'raceId': race_id, 'circuit_dnf_rate': dnf_rate})

circuit_dnf_df = pd.DataFrame(circuit_dnf_history)
df = df.merge(circuit_dnf_df, on='raceId', how='left')

# Végleges adattábla
final_df = df[[
    'raceId', 'name_race', 'year', 'decade', 'era', 'name_circuit', 'country', 'continent',
    'circuit_dnf_rate',
    'driverId', 'driver_name', 'driver_nationality', 'driver_all_races', 'experience_category',
    'constructorId', 'constructor_name', 'constructor_nationality',
    'position_x', 'position_y', 'grid', 'grid_category',
    'position_change', 'points', 'fastestLap', 'rank'
]].rename(columns={
    'name_race': 'race_name',
    'name_circuit': 'circuit_name',
    'position_x': 'quali_position',
    'position_y': 'race_position'
})

# Mentés
final_df.to_csv('f1_quali_vs_race.csv', index=False)

print(f"Kész! {len(final_df)} sor adattal.")
print("\nElső 5 sor:")
display(final_df.sample(5))
print("\nOszlopok:")
display(final_df.info())